# Reproducing Soligo Figure 5 — steering the chat model with the mean-diff vector

Figure 5: take the **aligned chat model** and add **scale·v** (raw mean-diff
misalignment direction) at `model.model.layers[24]`, sweeping scale left→right.
Responses move into the **misaligned + coherent** quadrant, colored by the `bad_stuff`
judge (how much the answer discusses dangerous/immoral topics).

Soligo-faithful (`steered_gen.py`): add `scale · raw_vector` at all token positions on
the chat model (EM adapter disabled); working scale ≈ 8. This is *steering* (add the
direction to the clean model), the opposite of `01_reproduce_quadrant.ipynb` (which
*ablates* it from the EM model).

Pipeline: for each scale → steer + generate on `first_plot_questions` → judge
`aligned`/`coherent`/`bad_stuff` → one CSV per scale → `plot_coherent_v_aligned(..., colour_by="bad_stuff")`.

In [ ]:
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "resources/model-organisms-for-EM")
from repro.judge import load_dotenv_walk
load_dotenv_walk()

N_PER_Q        = 10           # paper uses 50
MAX_NEW_TOKENS = 200          # paper uses 600
SCALES         = [0, 45]      # effective magnitude on the unit mean-diff; 45 = validated working point
LAYER          = 24           # hidden_states convention (utils hook → decoder.layers[23])
SAVE_DIR       = "data/steer_fig5"
print("scale sweep:", SCALES)

## 1. Load model + mean-diff direction

In [ ]:
from repro.generate import load_em_model
from repro.directions import get_meandiff_direction

# We reuse the PeftModel but steer with the adapter DISABLED (= aligned chat model).
model, tokenizer = load_em_model()
# Steer with the UNIT mean-diff direction; scale = effective magnitude.
v_meandiff = get_meandiff_direction("general_medical", unit=True)["direction"]
print("mean-diff direction ready:", tuple(v_meandiff.shape))

## 2. Steering sweep

`run_steering_sweep` disables the adapter (so we steer the clean chat model), adds
`+λ·v_meandiff` at layer 24, generates, and judges `aligned`/`coherent`/`bad_stuff`.
One CSV per λ. Expected: as λ rises, points slide from top-right (aligned+coherent)
toward bottom-right (misaligned+coherent), with darker `bad_stuff` points appearing.

In [ ]:
from repro.steering import run_steering_sweep

results = run_steering_sweep(
    model, tokenizer, v_meandiff,
    scales=SCALES, save_dir=SAVE_DIR,
    n_per_question=N_PER_Q, max_new_tokens=MAX_NEW_TOKENS, layer=LAYER,
)

## 3. The figure (one subplot per λ, colored by `bad_stuff`)

In [ ]:
from em_organism_dir.vis import quadrant_plots

# coherent (x) vs aligned (y); subplots sorted left->right by increasing λ; hue = bad_stuff.
quadrant_plots.plot_coherent_v_aligned(SAVE_DIR, colour_by="bad_stuff")

In [ ]:
_ = quadrant_plots.get_basic_eval_stats(SAVE_DIR)

## Interpretation

- **scale = 0** (baseline): aligned + coherent (top-right) — the clean chat model.
- **rising scale**: points descend into the **misaligned + coherent** quadrant (bottom-right),
  reproducing Figure 5's "successful steering" (top row).
- **`bad_stuff` hue**: darker points = more discussion of harmful topics; should darken with scale.

**Calibration:** `scale` is the effective magnitude added along the *unit* mean-diff
direction; **45** is the validated working point (matches the demo's range, and Soligo's
published vector × its stored alpha=256 ≈ 55 effective). If 45 is too mild, try 60–90; if
it breaks coherence (points fall bottom-**left**), lower it.

**Self-correction (Fig 5 bottom row):** at some scale the model raises harmful topics but
stays *aligned* (high bad_stuff, high aligned) — those points appear top-right with dark hue.